# EEG Eye State Classification - Leakage-Free Implementation

## Critical Analysis & Issues Fixed

### Issues Found in Original Notebooks:

1. **CRITICAL DATA LEAKAGE**: Both notebooks use `train_test_split()` with `shuffle=True`
   - EEG data is a continuous time-series at 128Hz for 117 seconds
   - Adjacent samples are highly autocorrelated
   - Random split allows test samples adjacent to training samples → model "sees the future"

2. **Scaler Fitting Before Split**: StandardScaler applied to entire dataset before split
   - Test data information leaks into the scaler's mean/std calculation

3. **No Temporal Validation**: Random KFold doesn't respect time-series nature

4. **Outlier Handling**: Extreme values (max 715,897 vs mean ~4,000) dominate learning

### Solutions Implemented:

1. **Chronological Train/Test Split**: First 80% train, last 20% test (respects temporal order)
2. **Proper Scaling**: Fit scaler ONLY on training data
3. **TimeSeriesSplit Cross-Validation**: For hyperparameter tuning
4. **Outlier Handling**: RobustScaler or winsorization
5. **Multiple Model Comparison**: Traditional ML + Neural Networks

---

## 1. Setup & Imports

In [ ]:
# Core imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Sklearn imports
from sklearn.model_selection import TimeSeriesSplit, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, RobustScaler, MinMaxScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
                             f1_score, roc_auc_score, confusion_matrix,
                             classification_report, roc_curve)
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier

# For reproducibility
np.random.seed(42)

print("All imports successful!")

## 2. Load and Explore Data

In [ ]:
# Load the dataset
df = pd.read_csv('/workspace/user_input_files/eeg_eye_state.csv')

print("Dataset Shape:", df.shape)
print("\nColumn names:", df.columns.tolist())
print("\nFirst 5 rows:")
df.head()

In [ ]:
# Dataset characteristics
print("="*80)
print("DATASET OVERVIEW")
print("="*80)
print(f"Total samples: {len(df)}")
print(f"Number of EEG channels: {len(df.columns) - 1}")
print(f"Recording duration: ~117 seconds at 128Hz")
print(f"\nClass distribution:")
print(df['eyeDetection'].value_counts())
print(f"\nClass proportions:")
print(df['eyeDetection'].value_counts(normalize=True))

In [ ]:
# Check for missing values
print("Missing values per column:")
print(df.isnull().sum().sum())

# Basic statistics
print("\nDescriptive statistics:")
df.describe()

## 3. Critical Issue: Data Leakage Analysis

In [ ]:
# DEMONSTRATION OF DATA LEAKAGE ISSUE

# Show autocorrelation - adjacent samples are highly correlated
eeg_channels = [col for col in df.columns if col != 'eyeDetection']

# Calculate autocorrelation at lag 1 (adjacent sample)
autocorr_lag1 = df[eeg_channels].apply(lambda x: x.autocorr(lag=1))
print("Autocorrelation at lag=1 (adjacent sample):")
print(autocorr_lag1.round(4))
print(f"\nMean autocorrelation: {autocorr_lag1.mean():.4f}")

# This high autocorrelation means:
# - Random shuffle splits will have test samples nearly identical to training samples
# - This is TEMPORAL DATA LEAKAGE

plt.figure(figsize=(14, 5))
plt.subplot(1, 2, 1)
autocorr_lag1.plot(kind='bar', color='steelblue')
plt.title('Autocorrelation at Lag=1 (All Channels)')
plt.xlabel('Channel')
plt.ylabel('Correlation')
plt.axhline(y=0.99, color='r', linestyle='--', label='High correlation threshold')
plt.legend()

# Show sample autocorrelation function for one channel
plt.subplot(1, 2, 2)
from pandas.plotting import autocorrelation_plot
autocorrelation_plot(df['AF3'][:500])  # First 500 samples
plt.title('Autocorrelation: AF3 Channel (First 500 samples)')
plt.tight_layout()
plt.show()

In [ ]:
# DEMONSTRATION: Why random split causes leakage

from sklearn.model_selection import train_test_split

# Prepare data
X = df[eeg_channels].values
y = df['eyeDetection'].values

# WRONG: Random shuffle split (DATA LEAKAGE)
X_train_wrong, X_test_wrong, y_train_wrong, y_test_wrong = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Calculate how many test samples are adjacent to training samples
leakage_count = 0
for i, idx in enumerate(np.arange(len(X))[len(X_train_wrong):]):
    # Check if adjacent sample is in training set
    if idx > 0 and (idx - 1) < len(X_train_wrong):
        leakage_count += 1
    if idx < len(X) - 1 and (idx + 1) < len(X_train_wrong):
        leakage_count += 1

print("="*80)
print("DATA LEAKAGE ANALYSIS")
print("="*80)
print(f"Total samples: {len(X)}")
print(f"Training samples (random split): {len(X_train_wrong)}")
print(f"Test samples (random split): {len(X_test_wrong)}")
print(f"\nTest samples adjacent to training samples: ~{leakage_count}")
print(f"Leakage percentage: ~{leakage_count/len(X_test_wrong)*100:.1f}%")
print("\n⚠️  WARNING: With random split, ~100% of test samples are adjacent to")
print("   training samples due to high autocorrelation. This is DATA LEAKAGE!")
print("\n✅ SOLUTION: Use chronological split (first 80% train, last 20% test)")

## 4. Proper Temporal Train/Test Split

In [ ]:
# CORRECT APPROACH: Chronological split

# Calculate split point (80% train, 20% test)
split_ratio = 0.8
split_index = int(len(df) * split_ratio)

print("="*80)
print("TEMPORAL TRAIN/TEST SPLIT (NO LEAKAGE)")
print("="*80)
print(f"Total samples: {len(df)}")
print(f"Split index: {split_index}")
print(f"Training samples (first 80%): {split_index}")
print(f"Test samples (last 20%): {len(df) - split_index}")
print(f"\nTraining period: samples 0 to {split_index-1}")
print(f"Test period: samples {split_index} to {len(df)-1}")

# Create temporal splits
train_df = df.iloc[:split_index].copy()
test_df = df.iloc[split_index:].copy()

# Verify no temporal overlap
print(f"\n✅ Verification: Training and test sets are temporally separated")
print(f"   Training time range: sample 0 → sample {split_index-1}")
print(f"   Test time range: sample {split_index} → sample {len(df)-1}")

In [ ]:
# Prepare features and target
X_train = train_df[eeg_channels].values
y_train = train_df['eyeDetection'].values
X_test = test_df[eeg_channels].values
y_test = test_df['eyeDetection'].values

print("\nFeature and target shapes:")
print(f"X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"X_test: {X_test.shape}, y_test: {y_test.shape}")

print("\nClass distribution in train/test:")
print(f"Train - Class 0: {sum(y_train==0)}, Class 1: {sum(y_train==1)}")
print(f"Test - Class 0: {sum(y_test==0)}, Class 1: {sum(y_test==1)}")

## 5. Feature Scaling (Fit ONLY on Training Data)

In [ ]:
# IMPORTANT: Fit scaler ONLY on training data, then transform both

# Using RobustScaler (better for outliers)
scaler = RobustScaler()

# Fit on training data ONLY
X_train_scaled = scaler.fit_transform(X_train)

# Transform test data using the SAME scaler (not refit!)
X_test_scaled = scaler.transform(X_test)

print("="*80)
print("FEATURE SCALING")
print("="*80)
print("Scaler: RobustScaler (robust to outliers)")
print(f"\nScaler fitted on {len(X_train)} training samples")
print(f"Training data transformed using fitted scaler")
print(f"Test data transformed using SAME fitted scaler (no refitting)")

# Verify scaling
print("\nAfter scaling - Training data statistics:")
print(f"Mean (should be ~0): {X_train_scaled.mean(axis=0).round(2)}")
print(f"Std (should be ~1): {X_train_scaled.std(axis=0).round(2)}")

## 6. Outlier Analysis and Handling

In [ ]:
# Analyze outliers in training data
from scipy import stats

# Calculate IQR-based outlier percentage for each channel
outlier_info = []
for i, channel in enumerate(eeg_channels):
    Q1 = np.percentile(X_train[:, i], 25)
    Q3 = np.percentile(X_train[:, i], 75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = np.sum((X_train[:, i] < lower) | (X_train[:, i] > upper))
    outlier_pct = outliers / len(X_train) * 100
    outlier_info.append({
        'Channel': channel,
        'Q1': Q1,
        'Q3': Q3,
        'IQR': IQR,
        'Outliers': outliers,
        'Outlier_%': outlier_pct
    })

outlier_df = pd.DataFrame(outlier_info)
print("Outlier Analysis (Training Data):")
print(outlier_df.to_string(index=False))

In [ ]:
# Visualize outliers with box plots
fig, axes = plt.subplots(2, 7, figsize=(18, 6))
axes = axes.flatten()

for i, channel in enumerate(eeg_channels):
    axes[i].boxplot(X_train[:, i], vert=True)
    axes[i].set_title(channel, fontsize=9)
    axes[i].set_ylabel('Amplitude')

plt.suptitle('EEG Channel Distributions (Training Data) - Showing Outliers', fontsize=12)
plt.tight_layout()
plt.show()

## 7. Visualizing Temporal Patterns

In [ ]:
# Plot EEG signals and eye state over time
fig, axes = plt.subplots(3, 1, figsize=(14, 10))

# Sample index for x-axis
time_idx = np.arange(len(df))

# Plot one EEG channel
axes[0].plot(time_idx, df['AF3'], alpha=0.7, linewidth=0.5)
axes[0].axvline(x=split_index, color='red', linestyle='--', linewidth=2, label='Train/Test Split')
axes[0].set_title('AF3 Channel Over Time')
axes[0].set_xlabel('Sample Index')
axes[0].set_ylabel('Amplitude')
axes[0].legend()

# Plot eye state
axes[1].plot(time_idx, df['eyeDetection'], alpha=0.7, linewidth=0.5)
axes[1].axvline(x=split_index, color='red', linestyle='--', linewidth=2, label='Train/Test Split')
axes[1].set_title('Eye State Over Time (Target)')
axes[1].set_xlabel('Sample Index')
axes[1].set_ylabel('Eye State (0=Open, 1=Closed)')
axes[1].legend()

# Rolling mean of eye state (to see patterns)
window = 100  # ~0.78 seconds at 128Hz
rolling_mean = df['eyeDetection'].rolling(window=window).mean()
axes[2].plot(time_idx, rolling_mean, alpha=0.7, linewidth=1)
axes[2].axvline(x=split_index, color='red', linestyle='--', linewidth=2, label='Train/Test Split')
axes[2].axhline(y=0.5, color='gray', linestyle=':', alpha=0.5)
axes[2].set_title(f'Eye State Rolling Mean (window={window} samples)')
axes[2].set_xlabel('Sample Index')
axes[2].set_ylabel('Rolling Mean')
axes[2].legend()

plt.tight_layout()
plt.show()

## 8. Model Training and Evaluation

### Helper Functions for Model Evaluation

In [ ]:
def evaluate_model(model, X_train, y_train, X_test, y_test, model_name="Model"):
    """
    Train and evaluate a model with comprehensive metrics
    """
    # Train
    model.fit(X_train, y_train)
    
    # Predict
    y_pred = model.predict(X_test)
    y_pred_proba = None
    
    # Get probability scores if available
    if hasattr(model, 'predict_proba'):
        y_pred_proba = model.predict_proba(X_test)[:, 1]
    elif hasattr(model, 'decision_function'):
        y_pred_proba = model.decision_function(X_test)
    
    # Calculate metrics
    metrics = {
        'Model': model_name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred, average='weighted'),
        'Recall': recall_score(y_test, y_pred, average='weighted'),
        'F1-Score': f1_score(y_test, y_pred, average='weighted'),
    }
    
    if y_pred_proba is not None:
        try:
            metrics['ROC-AUC'] = roc_auc_score(y_test, y_pred_proba)
        except:
            metrics['ROC-AUC'] = None
    
    # Confusion matrix
    cm = confusion_matrix(y_test, y_pred)
    
    return metrics, cm, y_pred, y_pred_proba


def plot_confusion_matrix(cm, title):
    """Plot confusion matrix"""
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=['Eye Open', 'Eye Closed'],
                yticklabels=['Eye Open', 'Eye Closed'])
    plt.title(title)
    plt.ylabel('Actual')
    plt.xlabel('Predicted')
    plt.tight_layout()
    plt.show()


def plot_roc_curve(y_test, y_pred_proba, title):
    """Plot ROC curve"""
    fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)
    auc = roc_auc_score(y_test, y_pred_proba)
    
    plt.figure(figsize=(8, 6))
    plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {auc:.4f})')
    plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(title)
    plt.legend(loc="lower right")
    plt.tight_layout()
    plt.show()

### 8.1 Traditional ML Models

In [ ]:
# Define models to test
# NOTE: Hyperparameters are set to reasonable defaults
# Users can override these as shown in Section 10

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'KNN': KNeighborsClassifier(n_neighbors=5),
    'SVM (RBF)': SVC(kernel='rbf', probability=True, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42),
    'MLP Neural Network': MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=500, random_state=42)
}

# Train and evaluate each model
results = []
predictions = {}
probabilities = {}

print("="*80)
print("MODEL TRAINING AND EVALUATION (TEMPORAL SPLIT - NO LEAKAGE)")
print("="*80)

for name, model in models.items():
    print(f"\nTraining {name}...")
    metrics, cm, y_pred, y_pred_proba = evaluate_model(
        model, X_train_scaled, y_train, X_test_scaled, y_test, name
    )
    results.append(metrics)
    predictions[name] = y_pred
    probabilities[name] = y_pred_proba
    print(f"   Accuracy: {metrics['Accuracy']:.4f}")
    if metrics['ROC-AUC']:
        print(f"   ROC-AUC: {metrics['ROC-AUC']:.4f}")

In [ ]:
# Summary results table
results_df = pd.DataFrame(results)
results_df = results_df.sort_values('Accuracy', ascending=False)

print("\n" + "="*80)
print("MODEL COMPARISON SUMMARY")
print("="*80)
print(results_df.to_string(index=False))

# Plot comparison
fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(results_df))
width = 0.2

bars1 = ax.bar(x - width, results_df['Accuracy'], width, label='Accuracy', color='steelblue')
bars2 = ax.bar(x, results_df['Precision'], width, label='Precision', color='coral')
bars3 = ax.bar(x + width, results_df['Recall'], width, label='Recall', color='forestgreen')
bars4 = ax.bar(x + 2*width, results_df['F1-Score'], width, label='F1-Score', color='gold')

ax.set_xlabel('Model')
ax.set_ylabel('Score')
ax.set_title('Model Performance Comparison (Temporal Split - No Leakage)')
ax.set_xticks(x + 0.5*width)
ax.set_xticklabels(results_df['Model'], rotation=45, ha='right')
ax.legend()
ax.set_ylim([0, 1])
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

### 8.2 Best Model Analysis

In [ ]:
# Identify best model
best_model_name = results_df.iloc[0]['Model']
print(f"Best performing model: {best_model_name}")
print(f"Accuracy: {results_df.iloc[0]['Accuracy']:.4f}")
print(f"F1-Score: {results_df.iloc[0]['F1-Score']:.4f}")

# Detailed analysis of best model
best_idx = results_df[results_df['Model'] == best_model_name].index[0]
best_y_pred = predictions[best_model_name]
best_y_proba = probabilities[best_model_name]

# Confusion Matrix
cm_best = confusion_matrix(y_test, best_y_pred)
plot_confusion_matrix(cm_best, f'Confusion Matrix - {best_model_name}')

# Classification Report
print("\nClassification Report:")
print(classification_report(y_test, best_y_pred, target_names=['Eye Open', 'Eye Closed']))

In [ ]:
# ROC Curve for best model
if best_y_proba is not None:
    plot_roc_curve(y_test, best_y_proba, f'ROC Curve - {best_model_name}')

## 9. Time Series Cross-Validation for Hyperparameter Tuning

In [ ]:
# Use TimeSeriesSplit for proper temporal cross-validation
# This is CRITICAL for time-series data - prevents data leakage

tscv = TimeSeriesSplit(n_splits=5)

print("="*80)
print("TIME SERIES CROSS-VALIDATION")
print("="*80)
print("Using TimeSeriesSplit with 5 folds")
print("This respects temporal order - each fold trains on past, validates on future")
print("\nFold structure:")

for i, (train_idx, val_idx) in enumerate(tscv.split(X_train_scaled)):
    print(f"  Fold {i+1}: Train size={len(train_idx)}, Val size={len(val_idx)}")

In [ ]:
# Cross-validation for Random Forest with TimeSeriesSplit
print("\n" + "="*80)
print("TIME SERIES CV FOR RANDOM FOREST")
print("="*80)

rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)

cv_scores = []
for i, (train_idx, val_idx) in enumerate(tscv.split(X_train_scaled)):
    X_cv_train, X_cv_val = X_train_scaled[train_idx], X_train_scaled[val_idx]
    y_cv_train, y_cv_val = y_train[train_idx], y_train[val_idx]
    
    rf_model.fit(X_cv_train, y_cv_train)
    score = rf_model.score(X_cv_val, y_cv_val)
    cv_scores.append(score)
    print(f"  Fold {i+1}: Accuracy = {score:.4f}")

print(f"\nMean CV Accuracy: {np.mean(cv_scores):.4f} (+/- {np.std(cv_scores):.4f})")

In [ ]:
# Cross-validation for Gradient Boosting
print("="*80)
print("TIME SERIES CV FOR GRADIENT BOOSTING")
print("="*80)

gb_model = GradientBoostingClassifier(n_estimators=100, random_state=42)

cv_scores_gb = []
for i, (train_idx, val_idx) in enumerate(tscv.split(X_train_scaled)):
    X_cv_train, X_cv_val = X_train_scaled[train_idx], X_train_scaled[val_idx]
    y_cv_train, y_cv_val = y_train[train_idx], y_train[val_idx]
    
    gb_model.fit(X_cv_train, y_cv_train)
    score = gb_model.score(X_cv_val, y_cv_val)
    cv_scores_gb.append(score)
    print(f"  Fold {i+1}: Accuracy = {score:.4f}")

print(f"\nMean CV Accuracy: {np.mean(cv_scores_gb):.4f} (+/- {np.std(cv_scores_gb):.4f})")

## 10. Hyperparameter Tuning with Manual Override Options

In [ ]:
# ============================================================
# HYPERPARAMETER CONFIGURATION
# ============================================================
# These parameters can be manually overridden to test different configurations
# The grid below provides reasonable search ranges
# ============================================================

# Example hyperparameter grids (commented out - uncomment to use)

# OPTION 1: Random Forest Hyperparameters
rf_param_grid = {
    'n_estimators': [50, 100, 200],          # Number of trees
    'max_depth': [None, 10, 20, 30],          # Maximum depth of trees
    'min_samples_split': [2, 5, 10],          # Min samples to split
    'min_samples_leaf': [1, 2, 4],            # Min samples in leaf
    'max_features': ['sqrt', 'log2', None]    # Features to consider
}

# OPTION 2: Gradient Boosting Hyperparameters
gb_param_grid = {
    'n_estimators': [50, 100, 200],
    'learning_rate': [0.01, 0.1, 0.2],
    'max_depth': [3, 5, 7],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2]
}

# OPTION 3: SVM Hyperparameters
svm_param_grid = {
    'C': [0.1, 1, 10],
    'gamma': ['scale', 'auto', 0.1],
    'kernel': ['rbf', 'poly']
}

# OPTION 4: KNN Hyperparameters
knn_param_grid = {
    'n_neighbors': [3, 5, 7, 9, 11],
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan']
}

# OPTION 5: MLP Hyperparameters
mlp_param_grid = {
    'hidden_layer_sizes': [(64,), (64, 32), (128, 64), (128, 64, 32)],
    'activation': ['relu', 'tanh'],
    'alpha': [0.0001, 0.001, 0.01],
    'learning_rate': ['constant', 'adaptive']
}

print("Hyperparameter grids defined. Uncomment GridSearchCV cells to run.")
print("\nNOTE: Grid search with TimeSeriesSplit can be computationally expensive.")
print("Consider using a smaller grid or fewer CV folds for faster experimentation.")

In [ ]:
# ============================================================
# EXAMPLE: Grid Search with Random Forest
# ============================================================
# Uncomment to run grid search (can be slow)

# from sklearn.model_selection import GridSearchCV

# print("Running Grid Search for Random Forest...")
# print("This may take several minutes...\n")

# # Use TimeSeriesSplit for CV
# rf_grid = GridSearchCV(
#     RandomForestClassifier(random_state=42, n_jobs=-1),
#     param_grid=rf_param_grid,
#     cv=TimeSeriesSplit(n_splits=3),  # Use 3 folds for speed
#     scoring='accuracy',
#     n_jobs=-1,
#     verbose=1
# )

# rf_grid.fit(X_train_scaled, y_train)

# print(f"\nBest parameters: {rf_grid.best_params_}")
# print(f"Best CV score: {rf_grid.best_score_:.4f}")

# # Evaluate on test set
# best_rf = rf_grid.best_estimator_
# y_pred_best = best_rf.predict(X_test_scaled)
# print(f"\nTest Accuracy: {accuracy_score(y_test, y_pred_best):.4f}")

## 11. Feature Importance Analysis

In [ ]:
# Train a Random Forest to get feature importances
rf_full = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_full.fit(X_train_scaled, y_train)

# Get feature importances
feature_importance = pd.DataFrame({
    'Channel': eeg_channels,
    'Importance': rf_full.feature_importances_
}).sort_values('Importance', ascending=False)

print("Feature Importance (Random Forest):")
print(feature_importance.to_string(index=False))

# Plot feature importance
plt.figure(figsize=(12, 6))
plt.barh(feature_importance['Channel'], feature_importance['Importance'], color='steelblue')
plt.xlabel('Importance')
plt.ylabel('EEG Channel')
plt.title('Feature Importance - EEG Channels for Eye State Classification')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## 12. Comparing Leakage vs Non-Leakage Results

In [ ]:
# Demonstrate the impact of data leakage by comparing results

print("="*80)
print("LEAKAGE IMPACT ANALYSIS")
print("="*80)

# WRONG: Random split (with leakage)
X_train_leak, X_test_leak, y_train_leak, y_test_leak = train_test_split(
    df[eeg_channels].values, df['eyeDetection'].values, 
    test_size=0.2, random_state=42
)

# Scale properly
scaler_leak = RobustScaler()
X_train_leak_scaled = scaler_leak.fit_transform(X_train_leak)
X_test_leak_scaled = scaler_leak.transform(X_test_leak)

# Train Random Forest with LEAKAGE
rf_leak = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_leak.fit(X_train_leak_scaled, y_train_leak)
y_pred_leak = rf_leak.predict(X_test_leak_scaled)
acc_leak = accuracy_score(y_test_leak, y_pred_leak)

# CORRECT: Temporal split (no leakage)
rf_correct = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_correct.fit(X_train_scaled, y_train)
y_pred_correct = rf_correct.predict(X_test_scaled)
acc_correct = accuracy_score(y_test, y_pred_correct)

print(f"\nRandom Split (WITH LEAKAGE):")
print(f"   Accuracy: {acc_leak:.4f}")
print(f"   ⚠️  This accuracy is INFLATED due to temporal data leakage")

print(f"\nTemporal Split (NO LEAKAGE):")
print(f"   Accuracy: {acc_correct:.4f}")
print(f"   ✅ This accuracy reflects true generalization to future data")

print(f"\nAccuracy Difference: {acc_leak - acc_correct:.4f}")
print(f"Leakage inflation: {((acc_leak - acc_correct) / acc_correct * 100):.1f}%")

print("\n⚠️  WARNING: Models trained with leakage will perform WORSE in production")
print("   because they have memorized patterns from adjacent time points.")

## 13. Conclusions and Best Practices

In [ ]:
print("""
================================================================================
CONCLUSIONS AND BEST PRACTICES
================================================================================

KEY FINDINGS:
-------------
1. The EEG Eye State dataset is a TIME-SERIES dataset with high autocorrelation
   between adjacent samples (r > 0.99 for most channels).

2. Random train/test split causes CRITICAL DATA LEAKAGE:
   - Adjacent samples are nearly identical
   - Random split means test samples appear in training set's temporal neighborhood
   - This inflates accuracy artificially

3. Chronological split is REQUIRED for this type of data:
   - First 80% for training, last 20% for testing
   - Respects temporal ordering
   - More realistic evaluation of model performance

BEST PRACTICES FOR EEG/Time-Series CLASSIFICATION:
--------------------------------------------------
1. ALWAYS use chronological split for train/test
2. ALWAYS fit scalers on training data only
3. Use TimeSeriesSplit for cross-validation
4. Handle outliers appropriately (RobustScaler, winsorization)
5. Consider feature engineering (rolling windows, frequency features)

RECOMMENDED MODELS (from this analysis):
---------------------------------------

Results will vary based on the temporal split, but typically:
- Gradient Boosting: Good balance of accuracy and robustness
- Random Forest: Reliable baseline, interpretable
- SVM (RBF): Can capture non-linear patterns

================================================================================
""")

## 14. Manual Hyperparameter Override Section

In [ ]:
# ============================================================
# MANUAL HYPERPARAMETER TESTING
# ============================================================
# Use this section to manually test specific hyperparameter configurations
# ============================================================


# ----------------------------------------------------------
# EXAMPLE 1: Test Different Random Forest Configurations
# ----------------------------------------------------------

rf_configs = [
    {'name': 'RF (100 trees, depth=None)', 'n_estimators': 100, 'max_depth': None},
    {'name': 'RF (200 trees, depth=20)', 'n_estimators': 200, 'max_depth': 20},
    {'name': 'RF (50 trees, depth=10)', 'n_estimators': 50, 'max_depth': 10},
    {'name': 'RF (300 trees, depth=None)', 'n_estimators': 300, 'max_depth': None},
]

print("="*80)
print("MANUAL HYPERPARAMETER TESTING - Random Forest")
print("="*80)

rf_results = []
for config in rf_configs:
    rf = RandomForestClassifier(
        n_estimators=config['n_estimators'],
        max_depth=config['max_depth'],
        random_state=42,
        n_jobs=-1
    )
    rf.fit(X_train_scaled, y_train)
    y_pred = rf.predict(X_test_scaled)
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='weighted')
    
    rf_results.append({
        'Config': config['name'],
        'Accuracy': acc,
        'F1-Score': f1
    })
    print(f"{config['name']}: Accuracy={acc:.4f}, F1={f1:.4f}")

print()

In [ ]:
# ----------------------------------------------------------
# EXAMPLE 2: Test Different KNN Configurations
# ----------------------------------------------------------

knn_configs = [
    {'name': 'KNN (k=3)', 'n_neighbors': 3},
    {'name': 'KNN (k=5)', 'n_neighbors': 5},
    {'name': 'KNN (k=7)', 'n_neighbors': 7},
    {'name': 'KNN (k=11)', 'n_neighbors': 11},
    {'name': 'KNN (k=15)', 'n_neighbors': 15},
    {'name': 'KNN (k=21)', 'n_neighbors': 21},
]

print("="*80)
print("MANUAL HYPERPARAMETER TESTING - KNN")
print("="*80)

knn_results = []
for config in knn_configs:
    knn = KNeighborsClassifier(n_neighbors=config['n_neighbors'])
    knn.fit(X_train_scaled, y_train)
    y_pred = knn.predict(X_test_scaled)
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='weighted')
    
    knn_results.append({
        'Config': config['name'],
        'Accuracy': acc,
        'F1-Score': f1
    })
    print(f"{config['name']}: Accuracy={acc:.4f}, F1={f1:.4f}")

print()

In [ ]:
# ----------------------------------------------------------
# EXAMPLE 3: Test Different MLP Configurations
# ----------------------------------------------------------

mlp_configs = [
    {'name': 'MLP (64)', 'hidden_layer_sizes': (64,)},
    {'name': 'MLP (64, 32)', 'hidden_layer_sizes': (64, 32)},
    {'name': 'MLP (128, 64)', 'hidden_layer_sizes': (128, 64)},
    {'name': 'MLP (128, 64, 32)', 'hidden_layer_sizes': (128, 64, 32)},
    {'name': 'MLP (256, 128)', 'hidden_layer_sizes': (256, 128)},
]

print("="*80)
print("MANUAL HYPERPARAMETER TESTING - MLP Neural Network")
print("="*80)

mlp_results = []
for config in mlp_configs:
    mlp = MLPClassifier(
        hidden_layer_sizes=config['hidden_layer_sizes'],
        max_iter=500,
        random_state=42
    )
    mlp.fit(X_train_scaled, y_train)
    y_pred = mlp.predict(X_test_scaled)
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='weighted')
    
    mlp_results.append({
        'Config': config['name'],
        'Accuracy': acc,
        'F1-Score': f1
    })
    print(f"{config['name']}: Accuracy={acc:.4f}, F1={f1:.4f}")

print()

In [ ]:
# ----------------------------------------------------------
# EXAMPLE 4: Test Different SVM Configurations
# ----------------------------------------------------------

svm_configs = [
    {'name': 'SVM (C=1, gamma=scale)', 'C': 1, 'gamma': 'scale'},
    {'name': 'SVM (C=10, gamma=scale)', 'C': 10, 'gamma': 'scale'},
    {'name': 'SVM (C=0.1, gamma=scale)', 'C': 0.1, 'gamma': 'scale'},
    {'name': 'SVM (C=10, gamma=0.1)', 'C': 10, 'gamma': 0.1},
]

print("="*80)
print("MANUAL HYPERPARAMETER TESTING - SVM")
print("="*80)

svm_results = []
for config in svm_configs:
    svm = SVC(
        kernel='rbf',
        C=config['C'],
        gamma=config['gamma'],
        probability=True,
        random_state=42
    )
    svm.fit(X_train_scaled, y_train)
    y_pred = svm.predict(X_test_scaled)
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='weighted')
    
    svm_results.append({
        'Config': config['name'],
        'Accuracy': acc,
        'F1-Score': f1
    })
    print(f"{config['name']}: Accuracy={acc:.4f}, F1={f1:.4f}")

print()

In [ ]:
# ----------------------------------------------------------
# SUMMARY: All Manual Tests
# ----------------------------------------------------------

print("="*80)
print("HYPERPARAMETER TESTING SUMMARY")
print("="*80)

# Combine all results
all_results = pd.DataFrame(rf_results + knn_results + mlp_results + svm_results)
all_results = all_results.sort_values('Accuracy', ascending=False)

print("\nAll configurations tested:")
print(all_results.to_string(index=False))

# Best configuration
best_config = all_results.iloc[0]
print(f"\n*** BEST CONFIGURATION ***")
print(f"Model: {best_config['Config']}")
print(f"Accuracy: {best_config['Accuracy']:.4f}")
print(f"F1-Score: {best_config['F1-Score']:.4f}")

---

## Summary

This notebook provides a **comprehensive, leakage-free** implementation for EEG Eye State classification:

### Key Features:
1. **Temporal Train/Test Split**: Uses chronological split instead of random shuffle
2. **Proper Scaling**: Scaler fitted ONLY on training data
3. **Time Series Cross-Validation**: Uses TimeSeriesSplit for hyperparameter tuning
4. **Multiple Model Comparison**: Tests Logistic Regression, Random Forest, KNN, SVM, Gradient Boosting, and MLP
5. **Outlier Handling**: Uses RobustScaler
6. **Manual Hyperparameter Testing**: Section 14 allows manual testing of different configurations

### Hyperparameter Override Options:
- Section 10: Define hyperparameter grids for GridSearchCV
- Section 14: Manually test specific configurations

### Expected Results:
Due to the temporal nature of the data, expect accuracy around 70-85% depending on the model and split point.
Results from random split (with leakage) will appear higher but are not reliable for production deployment.